Week 13 · Day 5 — Comparing GPT vs BERT Training Objectives
Why this matters

GPT and BERT are trained on different objectives: causal LM (next-word prediction) vs masked LM (fill-mask). Comparing them side by side helps you understand why GPT is good for generation and BERT for understanding.

Theory Essentials

Causal LM (GPT): learns to predict next token; optimized for continuation.

Masked LM (BERT): learns to recover missing tokens; optimized for context understanding.

Loss curves: show how quickly and smoothly models learn.

Outputs: GPT produces full text; BERT fills in gaps.

Comparing both highlights why we need different pretraining for different tasks.

In [1]:
# Setup
import torch
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    BertTokenizer, BertForMaskedLM,
    pipeline
)

# GPT-2 causal LM example
gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt_pipe = pipeline("text-generation", model=gpt_model, tokenizer=gpt_tok)

print("GPT completion:\n", gpt_pipe("The future of AI is", max_length=25)[0]["generated_text"])

# BERT masked LM example
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
bert_pipe = pipeline("fill-mask", model=bert_model, tokenizer=bert_tok)

print("BERT fill:\n", bert_pipe("The future of [MASK] is bright.")[0])


Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=25) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GPT completion:
 The future of AI is still far from determined. But until then, we need to take our eyes off the horizon, and think big.

—

"AI is a new frontier" is a theme from The New York Times Magazine.

[Image Credit: Chris Matyszczyk/Getty Images]


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


BERT fill:
 {'score': 0.10501672327518463, 'token': 14938, 'token_str': 'mankind', 'sequence': 'the future of mankind is bright.'}


1) Core (10–15 min)
Task: Run the code. Note the difference in output style.

2) Practice (10–15 min)
Task: Try both models with the theme “Sports.”

GPT prompt: "The soccer match ended"

BERT prompt: "The soccer [MASK] was exciting."

In [2]:
print("GPT completion:\n", gpt_pipe("The soccer match ended", max_length=25)[0]["generated_text"])

print("BERT fill:\n", bert_pipe("The soccer [MASK] was exciting.")[0])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=25) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GPT completion:
 The soccer match ended with a draw and a draw for the home side.

The Timbers managed to pull off a 3-1 win at home against the Portland Timbers, but it was a somewhat different match for the Timbers. For the first two games, the Timbers have conceded 12 goals. For the second two games, the Timbers have conceded 19 goals. For the third two games, the Timbers have conceded 11 goals.

The Timbers have won two of those matches with 3-1 margin.

In the third game against the Timbers, the Timbers have conceded 12 goals.

In the third game against the Timbers, the Timbers have conceded 11 goals.

In the fourth game against the Timbers, the Timbers have conceded 11 goals.

The Timbers have conceded 11 goals in the last two games against the Timbers.

In the fifth game against the Timbers, the Timbers have conceded 12 goals.

The Timbers have scored 12 goals in the last two games against the Timbers.

In the fourth game against the Timbers, the Timbers have conceded 10 goals.


3) Stretch (optional, 10–15 min)
Task: Fine-tune both on a tiny custom text file (like Day 2 and 3). Compare losses after 3 epochs.

In [1]:
# ---- Stretch (10–15 min): fine-tune both on a tiny text file and compare losses ----
from datasets import load_dataset
from transformers import (
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
    AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM
)

# 1) Tiny corpus (same for both models)
toy_text5 = """The future of AI is bright.
AI is transforming the world.
Dogs are friendly.
The cat sits on the mat.
"""
with open("tiny_corpus.txt", "w", encoding="utf-8") as f:
    f.write(toy_text5)

raw = load_dataset("text", data_files={"train": "tiny_corpus.txt"})

# 2) Helper: tokenize + chunk into fixed blocks
def chunkify(tokenizer, raw_ds, block_size=16):
    tok = raw_ds.map(lambda e: tokenizer(e["text"]), batched=True, remove_columns=["text"])
    def group_texts(examples):
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_len = (len(concatenated["input_ids"]) // block_size) * block_size
        return {
            k: [t[i:i+block_size] for i in range(0, total_len, block_size)]
            for k, t in concatenated.items()
        }
    lm_ds = tok["train"].map(group_texts, batched=True)
    # labels = input_ids for LM
    lm_ds = lm_ds.map(lambda b: {"labels": b["input_ids"]}, batched=True)
    return lm_ds

# 3) GPT-2 (causal LM)
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2_tok.pad_token = gpt2_tok.eos_token  # GPT-2 has no pad token
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
gpt2_ds = chunkify(gpt2_tok, raw)

gpt2_collator = DataCollatorForLanguageModeling(tokenizer=gpt2_tok, mlm=False)
gpt2_args = TrainingArguments(
    output_dir="./out_gpt2_toy",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=5,
    save_steps=1000,  # effectively don't save
    report_to=[],     # no TB/W&B
)
gpt2_trainer = Trainer(model=gpt2, args=gpt2_args, data_collator=gpt2_collator, train_dataset=gpt2_ds)
gpt2_result = gpt2_trainer.train()
gpt2_loss = gpt2_result.training_loss

# 4) BERT (masked LM)
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
bert_ds = chunkify(bert_tok, raw)

bert_collator = DataCollatorForLanguageModeling(tokenizer=bert_tok, mlm=True, mlm_probability=0.15)
bert_args = TrainingArguments(
    output_dir="./out_bert_toy",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=5,
    save_steps=1000,
    report_to=[],
)
bert_trainer = Trainer(model=bert, args=bert_args, data_collator=bert_collator, train_dataset=bert_ds)
bert_result = bert_trainer.train()
bert_loss = bert_result.training_loss

print(f"\n--- Loss after 3 epochs on the same tiny corpus ---")
print(f"GPT-2 (causal LM) loss: {gpt2_loss:.4f}")
print(f"BERT (MLM)          loss: {bert_loss:.4f}")


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


...

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss



--- Loss after 3 epochs on the same tiny corpus ---
GPT-2 (causal LM) loss: 3.4704
BERT (MLM)          loss: 2.3144



Mini-Challenge (≤40 min)

Build a notebook comparing GPT and BERT on the same theme.

Train each on ≥10 custom sentences.

Log loss values (just epoch avg is enough).

Generate at least 2 GPT completions and 2 BERT fills.

Write a short note on when GPT is better vs when BERT is better.

Acceptance Criteria

Shows both loss logs.

Has example outputs from both.

Includes 2–3 sentence reflection on differences.

In [2]:
# Mini-Challenge: GPT vs BERT, simple version

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments
)

torch.manual_seed(42)

# --- 1) Custom dataset (≥10 sentences) ---
sentences = [
    "AI helps people write faster.",
    "AI can translate between languages.",
    "Robots clean the house automatically.",
    "Voice assistants answer questions.",
    "Spam filters block unwanted emails.",
    "Recommendation systems suggest videos.",
    "Self-driving cars are in development.",
    "Chatbots provide customer support.",
    "Smart keyboards predict the next word.",
    "Image recognition detects objects."
]
dataset = Dataset.from_dict({"text": sentences * 5})  # repeat for training

# --- 2) GPT-2 (causal LM) ---
gpt_tok = AutoTokenizer.from_pretrained("gpt2")
gpt_tok.pad_token = gpt_tok.eos_token
gpt = AutoModelForCausalLM.from_pretrained("gpt2")

def tok_fn(examples): return gpt_tok(examples["text"], truncation=True, padding="max_length", max_length=32)
gpt_ds = dataset.map(tok_fn, batched=True)

gpt_collator = DataCollatorForLanguageModeling(tokenizer=gpt_tok, mlm=False)
gpt_args = TrainingArguments("./out_gpt", num_train_epochs=1, per_device_train_batch_size=2, logging_steps=5, report_to=[])

gpt_trainer = Trainer(model=gpt, args=gpt_args, train_dataset=gpt_ds, data_collator=gpt_collator)
gpt_result = gpt_trainer.train()
print("\nGPT loss:", gpt_result.training_loss)

# --- 3) BERT (masked LM) ---
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")

bert_ds = dataset.map(lambda e: bert_tok(e["text"], truncation=True, padding="max_length", max_length=32), batched=True)

bert_collator = DataCollatorForLanguageModeling(tokenizer=bert_tok, mlm=True)
bert_args = TrainingArguments("./out_bert", num_train_epochs=1, per_device_train_batch_size=2, logging_steps=5, report_to=[])

bert_trainer = Trainer(model=bert, args=bert_args, train_dataset=bert_ds, data_collator=bert_collator)
bert_result = bert_trainer.train()
print("BERT loss:", bert_result.training_loss)

# --- 4) Generate 2 GPT completions ---
prompts = ["AI can", "Robots will"]
for p in prompts:
    out = gpt.generate(**gpt_tok(p, return_tensors="pt"), max_new_tokens=15, pad_token_id=gpt_tok.eos_token_id)
    print("\nGPT completion:", gpt_tok.decode(out[0], skip_special_tokens=True))

# --- 5) Generate 2 BERT fills ---
masked_sents = ["Spam filters [MASK] unwanted emails.", "Smart keyboards predict the [MASK] word."]
for s in masked_sents:
    enc = bert_tok(s, return_tensors="pt")
    with torch.no_grad():
        logits = bert(**enc).logits
    mask_idx = (enc.input_ids[0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0].item()
    pred = logits[0, mask_idx].argmax().item()
    print("BERT fill:", s.replace("[MASK]", bert_tok.decode([pred])))

# --- 6) Reflection ---
print("\nReflection:")
print("GPT is better when we need fluent text continuations (generative tasks).")
print("BERT is better when we need precise word understanding or replacements (fill-mask, classification).")


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,4.827200
10,3.221500
15,2.748200
20,2.344400
25,1.958600



GPT loss: 3.0199707412719725


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,2.709000
10,1.542900
15,2.545800
20,5.580200
25,1.901700


BERT loss: 2.8559146881103517

GPT completion: AI can be used to write code.

The code is written in Python.

GPT completion: Robots will be able to track your movements.

The next time you're in
BERT fill: Spam filters filter unwanted emails.
BERT fill: Smart keyboards predict the next word.

Reflection:
GPT is better when we need fluent text continuations (generative tasks).
BERT is better when we need precise word understanding or replacements (fill-mask, classification).


Notes / Key Takeaways

GPT = autoregressive; BERT = bidirectional.

GPT → better for generation tasks.

BERT → better for understanding tasks.

Training objectives shape model behavior.

Loss curves help visualize learning efficiency.

Reflection

Why does GPT struggle to “fill in the middle” of a sentence?

Why does BERT struggle to generate long coherent text?

1) Why does GPT struggle to “fill in the middle” of a sentence?
GPT is trained with a causal objective: always predict the next token given everything before. It never learns to condition on both left and right context at once. So if you ask it to fill a blank in the middle, it doesn’t naturally know how to look ahead — it has to “fake it” by generating forward, which often breaks coherence.

2) Why does BERT struggle to generate long coherent text?
BERT is trained with a masked language modeling (MLM) objective: fill in missing tokens using both left and right context. It never learns the skill of generating text one token after another. When forced to generate, it tends to repeat, lose flow, or contradict itself because coherence over long spans was never part of its training.